In [ ]:
# 1) Imports e configuração
import warnings
import unicodedata
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

if "plotly_mimetype" in pio.renderers:
    pio.renderers.default = "plotly_mimetype"
elif "vscode" in pio.renderers:
    pio.renderers.default = "vscode"
elif "notebook_connected" in pio.renderers:
    pio.renderers.default = "notebook_connected"
elif "notebook" in pio.renderers:
    pio.renderers.default = "notebook"

In [ ]:
# 2) Parâmetros e caminhos
CSV_MAMOGRAFIA = Path("/home/anna/tcc/Tratamento/Mamografia/tabela_mamografia.csv")
CSV_MUNICIPIOS_REGIONAIS = Path("/home/anna/tcc/Tratamento/Municípios/tabela_municipios_regionais.csv")

ANOS = list(range(2013, 2026))
FAIXAS_ETARIAS = [
    "0-9",
    "10-19",
    "20-29",
    "30-39",
    "40-49",
    "50-59",
    "60-69",
    "70+",
]

TOP_N = 10

COR_BARRAS = "#2D9CDB"
COR_PONTOS = "#D7263D"

import json
import geobr
import plotly.express as px


def _normalizar_codigo_mapa(serie):
    return (
        pd.to_numeric(serie, errors="coerce")
        .round()
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.zfill(7)
    )

In [ ]:
def normalizar_texto(valor):
    texto = str(valor).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.split())


def formatar_inteiro(valor):
    try:
        return f"{int(valor):,}".replace(",", ".")
    except Exception:
        return str(valor)


def formatar_percentual(valor):
    try:
        return f"{float(valor):.2f}%".replace(".", ",")
    except Exception:
        return str(valor)


def encontrar_coluna(df, candidatos):
    mapa = {normalizar_texto(col): col for col in df.columns}
    for candidato in candidatos:
        chave = normalizar_texto(candidato)
        if chave in mapa:
            return mapa[chave]
    return None


def normalizar_codigo(serie):
    return (
        pd.to_numeric(serie, errors="coerce")
        .round()
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.zfill(7)
    )


def garantir_numerico(df, colunas):
    df = df.copy()
    existentes = [c for c in colunas if c in df.columns]
    if existentes:
        df[existentes] = df[existentes].apply(pd.to_numeric, errors="coerce").fillna(0)
    return df


def montar_base_longa(df):
    base = df[["Municipio", "Codigo", "Regional"]].copy()
    partes = []
    for ano in ANOS:
        for faixa in FAIXAS_ETARIAS:
            coluna = f"{ano}_{faixa}"
            if coluna not in df.columns:
                continue
            partes.append(
                base.assign(
                    ano=ano,
                    faixa_etaria=faixa,
                    exames=pd.to_numeric(df[coluna], errors="coerce").fillna(0),
                )
            )
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()


def montar_tabela_linhas(df_long):
    return (
        df_long.groupby(["ano", "faixa_etaria"], as_index=False)["exames"]
        .sum()
        .sort_values(["ano", "faixa_etaria"])
    )


def destacar_top(df, coluna_valor, n=TOP_N):
    return df.sort_values(coluna_valor, ascending=False).head(n).reset_index(drop=True)

In [ ]:
# 4) Carregamento e preparação da base
if not CSV_MAMOGRAFIA.exists():
    raise FileNotFoundError(f"Não encontrei o arquivo: {CSV_MAMOGRAFIA}")
if not CSV_MUNICIPIOS_REGIONAIS.exists():
    raise FileNotFoundError(f"Não encontrei o arquivo: {CSV_MUNICIPIOS_REGIONAIS}")

df = pd.read_csv(CSV_MAMOGRAFIA)

col_mun = encontrar_coluna(df, ["Municípios", "MUNICÍPIOS", "Municipios", "Município", "MUNICIPIO"])
col_cod = encontrar_coluna(df, ["Código", "CÓDIGO", "Codigo", "CODIGO"])

if col_mun is None or col_cod is None:
    raise KeyError("Não foi possível identificar as colunas de município e código em tabela_mamografia.csv.")

df = df.rename(columns={col_mun: "Municipio", col_cod: "Codigo"}).copy()
df["Municipio"] = df["Municipio"].astype(str).str.strip()
df["Codigo"] = normalizar_codigo(df["Codigo"])

ref_reg = pd.read_csv(CSV_MUNICIPIOS_REGIONAIS, dtype={"codigo_ibge": "string"})
ref_reg = ref_reg[["codigo_ibge", "regional_original"]].drop_duplicates().copy()
ref_reg["codigo_ibge"] = normalizar_codigo(ref_reg["codigo_ibge"])
ref_reg = ref_reg.rename(columns={"regional_original": "Regional"})

df = df.merge(ref_reg, left_on="Codigo", right_on="codigo_ibge", how="left").copy()
df["Regional"] = df["Regional"].fillna("Sem regional")
df = df.drop(columns=["codigo_ibge"], errors="ignore")

colunas_total_ano = [f"total_{ano}" for ano in ANOS if f"total_{ano}" in df.columns]
colunas_faixa_ano = [
    f"{ano}_{faixa}"
    for ano in ANOS
    for faixa in FAIXAS_ETARIAS
    if f"{ano}_{faixa}" in df.columns
]
colunas_numericas = colunas_total_ano + colunas_faixa_ano + (["total"] if "total" in df.columns else [])
df = garantir_numerico(df, colunas_numericas)

if not colunas_total_ano:
    raise KeyError("Não encontrei colunas total_YYYY na base de mamografia.")

df = df.assign(total_calculado_anos=df[colunas_total_ano].sum(axis=1))
if "total" in df.columns:
    df = df.assign(diferenca_total=df["total"] - df["total_calculado_anos"])
else:
    df = df.assign(diferenca_total=0)

df_long = montar_base_longa(df)

resumo_por_ano = pd.DataFrame({
    "Ano": ANOS,
    "Exames": [int(df[f"total_{ano}"].sum()) if f"total_{ano}" in df.columns else 0 for ano in ANOS],
})

resumo_por_faixa = (
    df_long.groupby("faixa_etaria", as_index=False)["exames"]
    .sum()
    .rename(columns={"faixa_etaria": "Faixa etária", "exames": "Exames"})
    .set_index("Faixa etária")
    .reindex(FAIXAS_ETARIAS)
    .reset_index()
    .fillna({"Exames": 0})
    .copy()
)

resumo_por_regional = (
    df.groupby("Regional", as_index=False)["total"]
    .sum()
    .rename(columns={"total": "Exames"})
    .sort_values("Exames", ascending=False)
    .reset_index(drop=True)
)

resumo_por_municipio = (
    df[["Municipio", "Codigo", "Regional", "total", "total_calculado_anos", "diferenca_total"]]
    .rename(columns={
        "Municipio": "Município",
        "Codigo": "Código",
        "total": "Exames",
        "total_calculado_anos": "Total calculado (anos)",
        "diferenca_total": "Diferença",
    })
    .sort_values("Exames", ascending=False)
    .reset_index(drop=True)
)

top_municipios = resumo_por_municipio.head(TOP_N).copy()
top_regionais = resumo_por_regional.head(TOP_N).copy()

tabela_diferencas = resumo_por_municipio[resumo_por_municipio["Diferença"] != 0].copy()

total_geral = int(df["total"].sum()) if "total" in df.columns else int(df["total_calculado_anos"].sum())
total_calculado = int(df["total_calculado_anos"].sum())
total_diferenca = int(df["diferenca_total"].abs().sum())

print(f"Base carregada: {len(df)} municípios")
print(f"Total geral de mamografias: {formatar_inteiro(total_geral)}")
print(f"Total calculado pela soma anual: {formatar_inteiro(total_calculado)}")
print(f"Diferença absoluta total entre 'total' e soma anual: {formatar_inteiro(total_diferenca)}")
print(f"Municípios com regional identificada: {int((df['Regional'] != 'Sem regional').sum())}")
print()
print("Top 5 municípios por volume de exames:")
display(top_municipios.head(5))

In [ ]:
# 5) Gráficos
resumo_por_ano_plot = resumo_por_ano.copy()
resumo_por_ano_plot["Ano"] = resumo_por_ano_plot["Ano"].astype(str)

fig_ano = go.Figure(
    go.Bar(
        x=resumo_por_ano_plot["Ano"],
        y=resumo_por_ano_plot["Exames"],
        marker_color=COR_BARRAS,
        text=[formatar_inteiro(v) for v in resumo_por_ano_plot["Exames"]],
        textposition="outside",
    )
)
fig_ano.update_layout(
    title=dict(text="Mamografias por ano", x=0.5),
    xaxis_title="Ano",
    yaxis_title="Quantidade de exames",
    template="plotly_white",
    height=500,
    width=1100,
    margin=dict(l=20, r=20, t=70, b=20),
)
fig_ano.show()

fig_faixa = go.Figure(
    go.Bar(
        x=resumo_por_faixa["Faixa etária"],
        y=resumo_por_faixa["Exames"],
        marker_color=COR_PONTOS,
        text=[formatar_inteiro(v) for v in resumo_por_faixa["Exames"]],
        textposition="outside",
    )
)
fig_faixa.update_layout(
    title=dict(text="Mamografias por faixa etária", x=0.5),
    xaxis_title="Faixa etária",
    yaxis_title="Quantidade de exames",
    template="plotly_white",
    height=500,
    width=1100,
    margin=dict(l=20, r=20, t=70, b=20),
)
fig_faixa.show()

df_tendencia = montar_tabela_linhas(df_long)
tendencia_pivot = (
    df_tendencia.pivot(index="ano", columns="faixa_etaria", values="exames")
    .fillna(0)
    .reindex(columns=FAIXAS_ETARIAS)
    .fillna(0)
)

cores_tendencia = [
    "#002F3E",
    "#5bc9e7",
    "#0c70d5",
    "#4a7189",
    "#DBB347",
    "#E79417",
    "#D01C08",
    "#9B3E29",
]

maior_valor_tendencia = int(tendencia_pivot.max().max()) if not tendencia_pivot.empty else 0
passo_tendencia = 25000
limite_quebra_tendencia = 1000
fator_compactacao_tendencia = 0.08

def _escala_tendencia(valor):
    valor = float(valor)
    if valor <= limite_quebra_tendencia:
        return valor
    return limite_quebra_tendencia + (valor - limite_quebra_tendencia) * fator_compactacao_tendencia

teto_tendencia = int(((maior_valor_tendencia + passo_tendencia - 1) // passo_tendencia) * passo_tendencia)
ticks_tendencia = [0, 1000] + list(range(25000, teto_tendencia + passo_tendencia, passo_tendencia))
ticks_tendencia = sorted(set(ticks_tendencia))
ticks_tendencia_plot = [_escala_tendencia(v) for v in ticks_tendencia]
limiar_proximidade = 12000
posicao_base_faixa = {
    faixa: ("top center" if idx % 2 == 0 else "bottom center")
    for idx, faixa in enumerate(FAIXAS_ETARIAS)
}

def _posicoes_texto_por_faixa(pivot_df, faixa):
    posicoes = []
    for i in range(len(pivot_df.index)):
        valor_atual = float(pivot_df.iloc[i][faixa])
        proximas = 0
        for outra_faixa in FAIXAS_ETARIAS:
            if outra_faixa == faixa or outra_faixa not in pivot_df.columns:
                continue
            valor_outra = float(pivot_df.iloc[i][outra_faixa])
            if abs(valor_atual - valor_outra) <= limiar_proximidade:
                proximas += 1

        base = posicao_base_faixa.get(faixa, "top center")
        if proximas >= 1:
            if (i + FAIXAS_ETARIAS.index(faixa)) % 2 == 0:
                posicoes.append("top center")
            else:
                posicoes.append("bottom center")
        else:
            posicoes.append(base)
    return posicoes

fig_tendencia = go.Figure()
for idx, faixa in enumerate(FAIXAS_ETARIAS):
    if faixa not in tendencia_pivot.columns:
        continue
    serie = tendencia_pivot[faixa]
    cor = cores_tendencia[idx % len(cores_tendencia)]
    posicoes_serie = _posicoes_texto_por_faixa(tendencia_pivot, faixa)
    # textos_serie = [formatar_inteiro(v) for v in serie]
    hover_serie = [f"{faixa}: {formatar_inteiro(v)}" for v in serie]
    fig_tendencia.add_trace(
        go.Scatter(
            x=tendencia_pivot.index.astype(str),
            y=[_escala_tendencia(v) for v in serie],
            mode="lines+markers",
            name=faixa,
            line=dict(color=cor, width=2.7),
            marker=dict(color=cor, size=9),
            hovertext=hover_serie,
            hovertemplate="%{x}<br>%{hovertext}<extra></extra>",
            cliponaxis=False,
        )
    )

fig_tendencia.update_layout(
    title=dict(text="Tendência anual por faixa etária", x=0.5),
    xaxis_title="Ano",
    yaxis_title="Quantidade de exames",
    template="plotly_white",
    height=680,
    width=1200,
    margin=dict(l=20, r=20, t=80, b=55),
    legend_title_text="Faixa etária",
    hovermode="x",
    yaxis=dict(
        tickmode="array",
        tickvals=ticks_tendencia_plot,
        ticktext=[formatar_inteiro(v) for v in ticks_tendencia],
        gridcolor="#E5ECF6",
        range=[0, _escala_tendencia(teto_tendencia)],
    ),
    xaxis=dict(tickangle=0),
)
fig_tendencia.show()

fig_municipios = go.Figure(
    go.Bar(
        x=top_municipios.sort_values("Exames")["Exames"],
        y=top_municipios.sort_values("Exames")["Município"],
        orientation="h",
        marker_color=COR_BARRAS,
        text=[formatar_inteiro(v) for v in top_municipios.sort_values("Exames")["Exames"]],
        textposition="outside",
    )
)
fig_municipios.update_layout(
    title=dict(text=f"Top {TOP_N} municípios por número de exames", x=0.5),
    xaxis_title="Quantidade de exames",
    yaxis_title="Município",
    template="plotly_white",
    height=600,
    width=1200,
    margin=dict(l=20, r=20, t=70, b=20),
)
fig_municipios.show()

fig_regionais = go.Figure(
    go.Bar(
        x=top_regionais.sort_values("Exames")["Exames"],
        y=top_regionais.sort_values("Exames")["Regional"],
        orientation="h",
        marker_color=COR_PONTOS,
        text=[formatar_inteiro(v) for v in top_regionais.sort_values("Exames")["Exames"]],
        textposition="outside",
    )
)
fig_regionais.update_layout(
    title=dict(text=f"Top {TOP_N} regionais por número de exames", x=0.5),
    xaxis_title="Quantidade de exames",
    yaxis_title="Regional",
    template="plotly_white",
    height=600,
    width=1200,
    margin=dict(l=20, r=20, t=70, b=20),
)
fig_regionais.show()

In [ ]:
# 6) Tabelas e resumos
resumo_geral = pd.DataFrame(
    {
        "Indicador": [
            "Municípios na base",
            "Total de mamografias",
            "Soma anual calculada",
            "Diferença absoluta total",
            "Municípios com regional identificada",
            "Ano com maior volume",
        ],
        "Valor": [
            len(df),
            total_geral,
            total_calculado,
            total_diferenca,
            int((df["Regional"] != "Sem regional").sum()),
            int(resumo_por_ano.loc[resumo_por_ano["Exames"].idxmax(), "Ano"]),
        ],
    }
)

display(resumo_geral)

display(
    resumo_por_ano.style.format({"Exames": formatar_inteiro})
)

display(
    resumo_por_faixa.style.format({"Exames": formatar_inteiro})
    .hide(axis="index")
 )

display(
    resumo_por_regional.head(15).style.format({"Exames": formatar_inteiro})
    .hide(axis="index")
 )

display(
    top_municipios[["Município", "Código", "Regional", "Exames", "Total calculado (anos)", "Diferença"]]
    .style.format({
        "Exames": formatar_inteiro,
        "Total calculado (anos)": formatar_inteiro,
        "Diferença": formatar_inteiro,
    })
    .hide(axis="index")
 )

if not tabela_diferencas.empty:
    print("Atenção: há diferença entre a coluna total e a soma anual em alguns municípios.")
    display(
        tabela_diferencas[["Município", "Código", "Exames", "Total calculado (anos)", "Diferença"]]
        .style.format({
            "Exames": formatar_inteiro,
            "Total calculado (anos)": formatar_inteiro,
            "Diferença": formatar_inteiro,
        })
        .hide(axis="index")
    )
else:
    print("✅ A coluna total coincide com a soma das colunas anuais para todos os municípios.")

print("✅ Gráficos, resumos e tabelas da base de mamografia gerados")

In [ ]:
# 6.1) Estatísticas por ano e participação por faixa etária
resumo_ano_faixa = (
    df_long.groupby(["ano", "faixa_etaria"], as_index=False)["exames"]
    .sum()
)

df_estatisticas_ano = (
    resumo_ano_faixa.groupby("ano", as_index=False)
    .agg(
        **{
            "Exames (soma das faixas)": ("exames", "sum"),
            "Média de exames (entre faixas)": ("exames", "mean"),
            "Desvio padrão (entre faixas)": ("exames", "std"),
        }
    )
    .rename(columns={"ano": "Ano"})
)

df_estatisticas_ano["Desvio padrão (entre faixas)"] = (
    df_estatisticas_ano["Desvio padrão (entre faixas)"]
    .fillna(0)
)

df_faixa_percentual = resumo_por_faixa[["Faixa etária", "Exames"]].copy()
total_faixas = df_faixa_percentual["Exames"].sum()
df_faixa_percentual["Percentual (%)"] = (
    (df_faixa_percentual["Exames"] / total_faixas) * 100
    if total_faixas > 0
    else 0
)

print("📊 Estatísticas por ano (soma das faixas, média e desvio padrão):")
display(
    df_estatisticas_ano.style.format(
        {
            "Exames (soma das faixas)": formatar_inteiro,
            "Média de exames (entre faixas)": lambda v: formatar_inteiro(round(v)),
            "Desvio padrão (entre faixas)": lambda v: formatar_inteiro(round(v)),
        }
    )
)

print("📊 Exames por faixa etária e percentual de participação:")
display(
    df_faixa_percentual.style.format(
        {
            "Exames": formatar_inteiro,
            "Percentual (%)": formatar_percentual,
        }
    ).hide(axis="index")
)

print("✅ DataFrames de estatísticas anuais e participação por faixa gerados")